# Brain tumour MRI classification (EfficientNetB0, transfer learning)

**Author:** Abdullah Ghani, ECE Paris (Data & AI)

4 classes: glioma, meningioma, pituitary tumour, no tumour. Public dataset: Kaggle `masoudnickparvar/brain-tumor-mri-dataset` (7,023 2D MRI slices; combines the figshare, SARTAJ and Br35H datasets).

Pipeline: download, stratified train/validation split, EfficientNetB0 (ImageNet) with frozen base then fine-tuning, evaluation on the dataset's separate Testing folder, confusion matrix, ROC curves, Grad-CAM.

> Research prototype. Not a medical device. Not for clinical use.

Runtime: Colab GPU (T4).

## 1. Setup

In [ ]:
!pip install -q kagglehub
import os, glob, json, random, numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.metrics import (confusion_matrix, classification_report, roc_auc_score,
                             roc_curve, f1_score, accuracy_score)
from sklearn.preprocessing import label_binarize
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 2. Data

In [ ]:
import kagglehub
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
TRAIN_DIR = glob.glob(os.path.join(path, "**", "Training"), recursive=True)[0]
TEST_DIR  = glob.glob(os.path.join(path, "**", "Testing"), recursive=True)[0]
CLASSES = sorted(os.listdir(TRAIN_DIR))
print("Classes:", CLASSES)
for d in [TRAIN_DIR, TEST_DIR]:
    print(os.path.basename(d), {c: len(os.listdir(os.path.join(d, c))) for c in CLASSES})

In [ ]:
IMG = 224; BATCH = 32
train_ds = tf.keras.utils.image_dataset_from_directory(TRAIN_DIR, labels="inferred", label_mode="int",
    class_names=CLASSES, image_size=(IMG, IMG), batch_size=BATCH, validation_split=0.15,
    subset="training", seed=SEED, color_mode="rgb")
val_ds = tf.keras.utils.image_dataset_from_directory(TRAIN_DIR, labels="inferred", label_mode="int",
    class_names=CLASSES, image_size=(IMG, IMG), batch_size=BATCH, validation_split=0.15,
    subset="validation", seed=SEED, color_mode="rgb")
test_ds = tf.keras.utils.image_dataset_from_directory(TEST_DIR, labels="inferred", label_mode="int",
    class_names=CLASSES, image_size=(IMG, IMG), batch_size=BATCH, shuffle=False, color_mode="rgb")
AUTO = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTO); val_ds = val_ds.prefetch(AUTO); test_ds = test_ds.prefetch(AUTO)

# Example images
plt.figure(figsize=(10, 4))
for x, y in test_ds.take(1):
    for i in range(8):
        plt.subplot(2, 4, i + 1); plt.imshow(x[i].numpy().astype("uint8")); plt.title(CLASSES[y[i]]); plt.axis("off")
plt.tight_layout(); plt.show()

## 3. Model: EfficientNetB0 (ImageNet) + light augmentation
The Keras EfficientNet includes its own input scaling, so images stay in the 0-255 range.

In [ ]:
augment = tf.keras.Sequential([
    layers.RandomRotation(0.05), layers.RandomZoom(0.1),
    layers.RandomTranslation(0.05, 0.05), layers.RandomContrast(0.1)], name="augment")

base = tf.keras.applications.EfficientNetB0(include_top=False, weights="imagenet", input_shape=(IMG, IMG, 3))
base.trainable = False
inp = layers.Input((IMG, IMG, 3))
x = augment(inp)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(len(CLASSES), activation="softmax")(x)
model = models.Model(inp, out)
model.compile(optimizer=optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

### 3.1 Phase 1: train the head (frozen base)

In [ ]:
es = callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=4, restore_best_weights=True)
h1 = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=[es])

### 3.2 Phase 2: fine-tune the top of the network

In [ ]:
base.trainable = True
for layer in base.layers[:-60]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
model.compile(optimizer=optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
es2 = callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=4, restore_best_weights=True)
h2 = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=[es2])

## 4. Learning curves

In [ ]:
def cat(k): return h1.history[k] + h2.history[k]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(cat("loss"), label="train"); ax[0].plot(cat("val_loss"), label="validation"); ax[0].set_title("Loss")
ax[1].plot(cat("accuracy"), label="train"); ax[1].plot(cat("val_accuracy"), label="validation"); ax[1].set_title("Accuracy")
for a in ax: a.axvline(len(h1.history["loss"]) - 0.5, ls="--", c="grey"); a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.savefig("learning_curve.png", dpi=150, bbox_inches="tight"); plt.show()

## 5. Evaluation on the held-out Testing folder

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_prob = model.predict(test_ds)
y_pred = y_prob.argmax(1)
yb = label_binarize(y_true, classes=range(len(CLASSES)))
auc_per_class = {c: float(roc_auc_score(yb[:, i], y_prob[:, i])) for i, c in enumerate(CLASSES)}
rep = classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True)
metrics = {
    "test_images": int(len(y_true)),
    "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
    "macro_f1": round(float(f1_score(y_true, y_pred, average="macro")), 4),
    "macro_auc": round(float(np.mean(list(auc_per_class.values()))), 4),
    "recall_per_class": {c: round(rep[c]["recall"], 4) for c in CLASSES},
    "precision_per_class": {c: round(rep[c]["precision"], 4) for c in CLASSES},
    "auc_per_class": {c: round(v, 4) for c, v in auc_per_class.items()},
    "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    "classes": CLASSES}
json.dump(metrics, open("metrics.json", "w"), indent=2)
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))
print(json.dumps(metrics, indent=2))
model.save("brain_tumor_effnetb0.keras")  # saved before Grad-CAM so a plotting error cannot lose the model

In [ ]:
cm = np.array(metrics["confusion_matrix"])
fig, ax = plt.subplots(figsize=(6, 5)); ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(CLASSES))); ax.set_yticks(range(len(CLASSES)))
ax.set_xticklabels(CLASSES, rotation=30); ax.set_yticklabels(CLASSES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion matrix (test set)")
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight"); plt.show()

plt.figure(figsize=(6, 5))
for i, c in enumerate(CLASSES):
    fpr, tpr, _ = roc_curve(yb[:, i], y_prob[:, i]); plt.plot(fpr, tpr, label=f"{c} (AUC {auc_per_class[c]:.3f})")
plt.plot([0, 1], [0, 1], "--", c="grey"); plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curves (one vs rest)"); plt.legend(loc="lower right")
plt.tight_layout(); plt.savefig("roc_curves.png", dpi=150, bbox_inches="tight"); plt.show()

## 6. Grad-CAM

In [ ]:
import cv2
LAST_CONV = "top_activation"
def grad_cam(img):
    eff = model.get_layer("efficientnetb0")
    gm = tf.keras.models.Model(eff.inputs, [eff.get_layer(LAST_CONV).output, eff.output])
    with tf.GradientTape() as tape:
        conv, feat = gm(img[None])
        probs = model.layers[-1](tf.reduce_mean(feat, axis=(1, 2)))
        k = tf.argmax(probs[0]); loss = probs[:, k]
    g = tape.gradient(loss, conv); w = tf.reduce_mean(g, axis=(0, 1, 2))
    cam = tf.maximum(tf.reduce_sum(conv[0] * w, -1), 0)
    return (cam / (tf.reduce_max(cam) + 1e-8)).numpy(), int(k)

x, y = next(iter(test_ds.unbatch().shuffle(500, seed=SEED).batch(4)))
fig, ax = plt.subplots(2, 4, figsize=(14, 7))
for i in range(4):
    img = x[i].numpy(); cam, k = grad_cam(tf.constant(img))
    heat = cv2.applyColorMap(np.uint8(255 * cv2.resize(cam, (IMG, IMG))), cv2.COLORMAP_JET)[:, :, ::-1]
    ax[0, i].imshow(img.astype("uint8")); ax[0, i].set_title(f"true: {CLASSES[int(y[i])]}"); ax[0, i].axis("off")
    ax[1, i].imshow(np.uint8(0.6 * img + 0.4 * heat)); ax[1, i].set_title(f"pred: {CLASSES[k]}"); ax[1, i].axis("off")
plt.tight_layout(); plt.savefig("gradcam_examples.png", dpi=150, bbox_inches="tight"); plt.show()

## 7. Save the model and results

In [ ]:
!zip -q brain_tumor_results.zip brain_tumor_effnetb0.keras metrics.json learning_curve.png confusion_matrix.png roc_curves.png gradcam_examples.png
print("Saved brain_tumor_results.zip")

## Limitations
- This public dataset merges three sources and contains near-duplicate slices; some may appear in both the Training and Testing folders, which would make the test metrics optimistic.
- Slices are classified one by one, without patient-level information or 3D context.
- Mixed sources and MRI sequences, and no external validation.